## Feature Engineering

**Fuel lag features**

Airlines don't react to this month's fuel price. They react to trends over the past quarter. Add fuel prices from the past three months:

In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/model_ready_no_covid.csv')
df = df.sort_values(['country', 'year_month']).reset_index(drop=True)

df['fuel_lag1'] = df.groupby('country')['jet_fuel_usd_per_gallon'].shift(1)
df['fuel_lag2'] = df.groupby('country')['jet_fuel_usd_per_gallon'].shift(2)
df['fuel_lag3'] = df.groupby('country')['jet_fuel_usd_per_gallon'].shift(3)

**Fuel change features**

A sudden spike is more disruptive than a gradual rise.

In [2]:
df['fuel_change_1m'] = df.groupby('country')['jet_fuel_usd_per_gallon'].pct_change(1) * 100
df['fuel_change_3m'] = df.groupby('country')['jet_fuel_usd_per_gallon'].pct_change(3) * 100
df['fuel_change_6m'] = df.groupby('country')['jet_fuel_usd_per_gallon'].pct_change(6) * 100
df['fuel_volatility_6m'] = df.groupby('country')['jet_fuel_usd_per_gallon'].transform(
    lambda x: x.rolling(6).std()
)

**Seasonal features**

Month of year matters significantly for air travel.

In [3]:
df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month'] = df['year_month_dt'].dt.month
df['quarter'] = df['year_month_dt'].dt.quarter

# Peak travel months (June-August, December)
df['is_peak_season'] = df['month'].isin([6, 7, 8, 12]).astype(int)

In [4]:
import numpy as np

# Nullify change features for first 6 months after COVID gap
covid_end = pd.Timestamp('2022-03-01')
mask = (df['year_month_dt'] > covid_end) & \
       (df['year_month_dt'] <= covid_end + pd.DateOffset(months=6))

df.loc[mask, ['fuel_change_1m', 'fuel_change_3m',
              'fuel_change_6m', 'fuel_volatility_6m']] = np.nan

**GDP change feature**

To determine whether a country's economy is growing or shrinking.

In [5]:
df['gdp_yoy_change'] = df.groupby('country')['gdp_per_capita'].pct_change(12) * 100

**Encoding the LCC flag**

In [6]:
import pandas as pd

routes = pd.read_csv('../data/processed/routes.csv')
airline_types = pd.read_csv('../data/processed/airline_classifications_clean.csv')

routes_enriched = routes.merge(airline_types, on='airline_iata', how='left')

country_features = routes_enriched.groupby('destination_country').agg(
    total_routes=('airline_iata', 'count'),
    lcc_routes=('carrier_type', lambda x: (x == 'LCC').sum())
).reset_index()

country_features['lcc_share'] = country_features['lcc_routes'] / country_features['total_routes']

df = df.merge(country_features, left_on='country', right_on='destination_country', how='left')

# now this will work
df['is_high_lcc_corridor'] = (df['lcc_share'] > 0.4).astype(int)

In [7]:
df['is_high_lcc_corridor'] = (df['lcc_share'] > 0.4).astype(int)

**Final feature set**

In [8]:
feature_cols = [
    'jet_fuel_usd_per_gallon',
    'fuel_lag1', 'fuel_lag2', 'fuel_lag3',
    'fuel_change_3m', 'fuel_change_6m',
    'fuel_volatility_6m',
    'gdp_per_capita', 'gdp_yoy_change',
    'lcc_share',
    'month', 'quarter', 'is_peak_season'
]
target_col = 'log_arrivals_diff'

df_final = df[feature_cols + [target_col, 'country', 'year_month']].dropna()
print(f"Final dataset rows: {len(df_final)}")
print(f"Countries: {df_final['country'].nunique()}")
print(f"\nFeature set:")
for f in feature_cols:
    print(f"  - {f}: {df_final[f].describe()[['mean','std','min','max']].to_dict()}")
df_final.to_csv('../data/processed/final_features.csv', index=False)
print("Saved final_features.csv")

Final dataset rows: 1576
Countries: 8

Feature set:
  - jet_fuel_usd_per_gallon: {'mean': 2.245890355329949, 'std': 0.6506804287837151, 'min': 0.9456, 'max': 3.95825}
  - fuel_lag1: {'mean': 2.2530459390862942, 'std': 0.6538703106468466, 'min': 0.9456, 'max': 3.95825}
  - fuel_lag2: {'mean': 2.2590832487309647, 'std': 0.6582622886115422, 'min': 0.9456, 'max': 3.95825}
  - fuel_lag3: {'mean': 2.2693131979695433, 'std': 0.6640290336956531, 'min': 0.9456, 'max': 3.95825}
  - fuel_change_3m: {'mean': 0.4497323391611305, 'std': 15.802301005414105, 'min': -59.491351509821165, 'max': 43.13764379214597}
  - fuel_change_6m: {'mean': 1.0343441638320554, 'std': 22.463645190809604, 'min': -64.30739425250242, 'max': 51.323712128452655}
  - fuel_volatility_6m: {'mean': 0.1881302159043381, 'std': 0.15006886644604006, 'min': 0.03100663907398853, 'max': 1.0011470421471567}
  - gdp_per_capita: {'mean': 24154.63738441369, 'std': 19221.264492293743, 'min': 1426.14649179546, 'max': 56103.7323182554}
  - gd